# Generation checks

Full pipeline test from a raw guest question through to a generated answer, with an LLM doing
the query understanding as well as the final reply -- two calls per question, both fully
printed:

1. **Understand** -- one structured-output LLM call reads the question and returns intent
   (`menu` vs `faq`) plus retrieval filters (dietary, price ceiling, allergens to exclude).
   This replaces `01_retrieval_checks.ipynb`'s regex-based `parse_constraints()` and this
   notebook's earlier top-1-`item_type` `classify_intent()` -- both were explicitly called out
   as heuristics to swap for an LLM call once one was available.
2. **Retrieve** -- the rest of `01`'s pipeline, unchanged: hybrid `alpha=0.75` -> allergen
   exclude (union of `allergens_contains`/`allergens_may_contain`) -> rerank `rerank-v3.5` ->
   gate `0.15`.
3. **Generate** -- a grounding-only system prompt, with a tone instruction picked from the
   understanding step's intent: **precise and literal for menu facts**
   (price/allergens/nutrition must stay exact), **warm and conversational for FAQ answers**
   (house-policy copy can be phrased more naturally). Unlike Gemini's 3.x Flash line (this
   notebook's previous backend, which silently ignored `temperature`/`top_p`/`top_k`), Groq's
   models genuinely respect `temperature` -- confirmed directly (same prompt at `temperature=0`
   came back identical three times in a row; at `temperature=1.8` it varied every time). Section
   5 now pairs the tone instruction with a real per-intent `temperature`, rather than using
   prompt text as a `temperature` substitute.

Every test cell prints both full prompts (understanding and generation) and both LLM outputs,
so the whole path from question to answer is inspectable, not just the final text.

Runs against the live Weaviate collection and the live LLM API -- nothing is mocked.

Prerequisites: the collection is loaded (`scripts/load_knowledge_base.py`), and `.env` holds
`WEAVIATE_URL` / `WEAVIATE_API_KEY`, `EMBEDDING_API_KEY` (Cohere, for reranking), and
`GROQ_API_KEY` or `LLM_API_KEY` (Gemini), matching `LLM_PROVIDER` (`groq` by default; see
section 2) -- `00_environment_check.ipynb` verifies the Groq path. Reranking needs the Cohere key;
a trial key is rate-limited, so some cells may fall back to hybrid order (see `01`). Without
an LLM key configured for the active provider, the understanding step falls back to `{intent: "menu", no filters}` and
generation is skipped, so retrieval-only checks still run.

## Pipeline overview

What happens between a guest's question and the answer -- every external API call, in order.
Two LLM calls (provider per `LLM_PROVIDER`, section 2) and up to two Cohere calls per question (the embedding call is made by
Weaviate itself, mid-retrieval, and isn't shown as a separate box the way the direct rerank
call is -- see section 4's note on why it can't be measured from here).

Node names match the actual function each step calls -- see the matching section below for
what each one does.

```mermaid
flowchart LR
    Q[Guest question] --> U["understand_query()<br/>LLM 1 (LLM_PROVIDER)"]
    U --> F[build_filter]
    F --> R["retrieve()<br/>Weaviate hybrid"]
    R --> EX{allergens_exclude?}
    EX -- yes --> DROP[drop excluded rows]
    EX -- no --> KEEP[keep all rows]
    DROP --> RR
    KEEP --> RR["rerank()<br/>Cohere"]
    RR --> GATE{"top score >= 0.15?"}
    GATE -- no --> NC["CONTEXT = no match"]
    GATE -- yes --> CTX[build_context]
    NC --> PROMPT
    CTX --> PROMPT[build_user_prompt]
    PROMPT --> TEMP[tone_for intent]
    TEMP --> GEN["call_llm()<br/>LLM 2 (LLM_PROVIDER)"]
    GEN --> ANS[Answer to guest]
```

## 1. Connect

Loads `.env`, connects to Weaviate Cloud (same as `01`), and reads the LLM credentials
(`00_environment_check.ipynb` verifies these are valid). The client stays open for the whole
notebook, including the demo cell at the end -- re-run this cell to reconnect if the kernel
session drops. Closes any `client` left over from a previous run of this same cell first --
otherwise reconnecting in an already-running kernel leaks the old connection's sockets until
Python's garbage collector eventually finalizes them, which is what an "unclosed
`<ssl.SSLSocket ...>`" `ResourceWarning` after a reconnect actually is.

In [ ]:
import contextlib
import json
import math
import os
import random
import time
import urllib.error
import urllib.request

import weaviate
from dotenv import find_dotenv, load_dotenv
from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, FilterReturn, MetadataQuery

load_dotenv(find_dotenv(usecwd=True))

PROVIDER = os.environ.get("EMBEDDING_PROVIDER", "cohere").lower()
COHERE_KEY = os.environ.get("EMBEDDING_API_KEY", "")
_hdr = "X-OpenAI-Api-Key" if PROVIDER == "openai" else "X-Cohere-Api-Key"

# Re-running this cell in an already-running kernel would otherwise leave the previous
# client's sockets open until Python's garbage collector gets around to them -- that's what
# an "unclosed <ssl.SSLSocket ...>" ResourceWarning after a reconnect is. Close it first.
# globals().get(...) (rather than referencing `client` directly) means this doesn't depend on
# `client` already existing in this kernel session.
_previous_client = globals().get("client")
if _previous_client is not None:
    with contextlib.suppress(Exception):
        _previous_client.close()

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={_hdr: COHERE_KEY},
)
kb = client.collections.get("KnowledgeBase")
print("connected -", kb.aggregate.over_all(total_count=True).total_count, "objects")

# LLM_PROVIDER selects which of the two call_llm() code paths below actually runs -- "groq"
# (OpenAI-compatible chat/completions) or "gemini" (generateContent). Edit directly, or set
# LLM_PROVIDER in .env, to switch. See section 2's markdown for why each needs its own path
# rather than one shared request shape.
LLM_PROVIDER = os.environ.get("LLM_PROVIDER", "groq").lower()

if LLM_PROVIDER == "gemini":
    LLM_API_KEY = os.environ.get("LLM_API_KEY", "")
    UNDERSTAND_MODEL = os.environ.get("LLM_MODEL", "gemini-3.8-flash")
    GENERATION_MODEL = UNDERSTAND_MODEL
else:
    LLM_API_KEY = os.environ.get("GROQ_API_KEY", "")
    UNDERSTAND_MODEL = os.environ.get("UNDERSTAND_MODEL", "openai/gpt-oss-120b")
    GENERATION_MODEL = os.environ.get("GENERATION_MODEL", "openai/gpt-oss-120b")

print(f"LLM_PROVIDER     = {LLM_PROVIDER}")
print(f"UNDERSTAND_MODEL = {UNDERSTAND_MODEL}")
print(f"GENERATION_MODEL = {GENERATION_MODEL}")
_key_status = f"set ({len(LLM_API_KEY)} chars)" if LLM_API_KEY else "NOT SET"
print(f"LLM_API_KEY      = {_key_status}")
if not LLM_API_KEY:
    print(
        f"warning: no API key set for LLM_PROVIDER={LLM_PROVIDER!r} -- "
        "understanding/generation cells will use fallbacks"
    )

## 2. LLM call helper

One thin wrapper around whichever provider `LLM_PROVIDER` selects -- Groq's OpenAI-compatible
`chat/completions` endpoint, or Gemini's `generateContent` REST endpoint (same style
`00_environment_check.ipynb` uses for its credential check) -- shared by both LLM calls this
notebook makes: query understanding (structured JSON, via `response_schema`) and answer
generation (free text). Keeping one call site per concern means both share the same error
handling, timeout, retry, and rotation behavior regardless of which provider is active,
instead of drifting apart the way this notebook and its Gemini-only predecessor once did.

The two providers genuinely differ, not just in URL/auth shape:

- **Groq** genuinely applies `temperature` (confirmed directly: `temperature=0` is
  deterministic, higher values vary) and supports `reasoning_effort` (`"low"`/`"medium"`/
  `"high"`, gpt-oss models only, controlling how many reasoning tokens the model spends before
  answering).
- **Gemini**'s 3.x Flash line silently ignores `temperature`/`top_p`/`top_k` and has no
  `reasoning_effort` equivalent -- the API accepts them, returns `200`, and does nothing with
  them (Google's own confirmed behavior). When `LLM_PROVIDER == "gemini"`, `call_llm()` drops
  both parameters rather than sending something the provider silently no-ops; section 5's tone
  instruction is what actually steers Gemini's output instead.

`call_llm()` returns `{"text": ..., "usage": {...}}` either way -- both providers' own token
counts (Groq's `usage`, Gemini's `usageMetadata`) are read directly rather than estimated from
string length, which is what lets `show_answer()` (section 8) report exactly how many tokens
each question consumed.

Two layers of rate-limit handling, not just one, regardless of provider:

- **Proactive pacing** -- `_pace_llm_call()` blocks just long enough before every request to
  keep the call rate under `LLM_MAX_RPM`, which is itself provider-dependent (Groq's
  confirmed per-key quota is far higher than Gemini's free-tier per-minute cap -- see the code
  cell below). Edit the constant directly to match your key's actual quota.
- **Reactive retry**, for whatever pacing doesn't prevent (a burst from another process on the
  same key, a transient 5xx, a dropped connection): 429/5xx/connection errors retry with
  full-jitter exponential backoff, honoring a `Retry-After` header when the API sends one. A
  non-retryable error (400 bad request, 401/403 auth) raises immediately.

In [ ]:
RETRYABLE_HTTP_CODES = {408, 429, 500, 502, 503, 504}
MAX_LLM_RETRIES = 5
BASE_DELAY_S = 1.0
MAX_DELAY_S = 20.0
# Both figures are this key's own confirmed per-provider quota, not a docs-page number (docs
# pages read lower/stale for a given tier) -- edit to match your own key's actual limit.
LLM_MAX_RPM = 1000.0 if LLM_PROVIDER == "groq" else 15.0

# Free-tier request limits, confirmed 2026-09-18 -- a *local, proactive* guard, separate from
# LLM_MAX_RPM's reactive pacing above. RATE_LIMITS backs _key_available()/_record_key_usage()
# in the key-pool cell below: a pool key already at its own confirmed limit is skipped with no
# HTTP request made, rather than tried and left to 429. Edit if your tier differs.
RATE_LIMITS = {
    "groq": {"rpm": 30, "rpd": 1000},
    "gemini": {"rpm": 15, "rpd": 1500},
}

_last_llm_call_at = 0.0

GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"
# Cloudflare (in front of api.groq.com) 403s Python's default "Python-urllib/x.y" User-Agent
# with body "error code: 1010" -- easy to mistake for an auth failure. A normal-looking
# User-Agent avoids it.
_GROQ_HEADERS = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0 (compatible; wagami-rag-notebook/1.0)",
}


class AllKeysRateLimitedError(Exception):
    """Every pool key is at its own local RPM/RPD limit -- raised without making any HTTP
    request, rather than sending a call the provider would just reject. See the key-pool
    cell's _key_available()/_record_key_usage()."""


def _zero_usage() -> dict:
    """A fresh {prompt,completion,total: 0} usage dict, for calls that never hit the API."""
    return {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}


def _http_error_detail(e: urllib.error.HTTPError) -> str:
    """Read and return a truncated response body from an HTTPError, then close it.

    A 429's body is where the actual reason lives -- both providers' error responses name the
    specific limit that was hit (Groq: {"error": {"code": "rate_limit_exceeded", ...}}; Gemini:
    a QuotaFailure naming the exceeded quota ID), which the bare status code and reason phrase
    never show.
    """
    try:
        raw = e.read().decode("utf-8", errors="replace")
    except OSError:
        raw = ""
    e.close()
    return raw[:400].replace("\n", " ")


def _pace_llm_call() -> None:
    """Block just long enough to keep call_llm() under LLM_MAX_RPM requests/minute."""
    global _last_llm_call_at
    min_interval = 60.0 / LLM_MAX_RPM
    wait = min_interval - (time.monotonic() - _last_llm_call_at)
    if wait > 0:
        time.sleep(wait)
    _last_llm_call_at = time.monotonic()


def _call_llm_single_key(
    api_key: str,
    system_prompt: str,
    user_prompt: str,
    model: str,
    response_schema: dict | None,
    temperature: float | None,
    reasoning_effort: str | None,
    max_retries: int,
) -> dict:
    """The actual HTTP call, against whichever provider LLM_PROVIDER selects, for one key.

    Not called directly by the rest of the notebook -- call_llm() (rebound to
    _call_llm_with_rotation by the key-pool cell below) wraps this with pooling/rotation.
    """
    is_gemini = LLM_PROVIDER == "gemini"
    if is_gemini:
        url = (
            f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
            f"?key={api_key.strip()}"
        )
        generation_config: dict = {}
        if response_schema is not None:
            generation_config["responseMimeType"] = "application/json"
            generation_config["responseSchema"] = response_schema
        body = json.dumps(
            {
                "systemInstruction": {"parts": [{"text": system_prompt}]},
                "contents": [{"role": "user", "parts": [{"text": user_prompt}]}],
                "generationConfig": generation_config,
            }
        ).encode()
        headers = {"Content-Type": "application/json"}
    else:
        payload: dict = {
            "model": model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        }
        if temperature is not None:
            payload["temperature"] = temperature
        if reasoning_effort is not None:
            payload["reasoning_effort"] = reasoning_effort
        if response_schema is not None:
            payload["response_format"] = {
                "type": "json_schema",
                "json_schema": {
                    "name": "response",
                    "strict": True,
                    "schema": response_schema,
                },
            }
        body = json.dumps(payload).encode()
        headers = _GROQ_HEADERS

    delay_cap = BASE_DELAY_S
    for attempt in range(1, max_retries + 1):
        _pace_llm_call()
        url_or_groq = url if is_gemini else GROQ_URL
        req = urllib.request.Request(url_or_groq, data=body, headers=headers, method="POST")
        if not is_gemini:
            req.add_header("Authorization", f"Bearer {api_key.strip()}")
        retry_after: str | None = None
        detail = ""
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = json.load(resp)
            if is_gemini:
                parts = data["candidates"][0]["content"]["parts"]
                text = " ".join(p.get("text", "") for p in parts).strip()
                meta = data.get("usageMetadata", {})
                usage = {
                    "prompt_tokens": meta.get("promptTokenCount", 0),
                    "completion_tokens": meta.get("candidatesTokenCount", 0),
                    "total_tokens": meta.get("totalTokenCount", 0),
                }
            else:
                text = data["choices"][0]["message"]["content"].strip()
                meta = data.get("usage", {})
                usage = {
                    "prompt_tokens": meta.get("prompt_tokens", 0),
                    "completion_tokens": meta.get("completion_tokens", 0),
                    "total_tokens": meta.get("total_tokens", 0),
                }
            return {"text": text, "usage": usage}
        except urllib.error.HTTPError as e:
            retryable = e.code in RETRYABLE_HTTP_CODES
            retry_after = e.headers.get("Retry-After") if retryable else None
            label = f"HTTP {e.code} {e.reason}"
            detail = _http_error_detail(e)
            if not retryable or attempt == max_retries:
                print(f"   [LLM {label}, giving up -- {detail}]")
                raise
        except urllib.error.URLError as e:
            label = f"connection error ({e.reason})"
            if attempt == max_retries:
                raise

        if retry_after:
            try:
                wait = float(retry_after)
            except ValueError:
                wait = random.uniform(0, delay_cap)
        else:
            wait = random.uniform(0, delay_cap)
        suffix = f"  {detail}" if detail else ""
        print(f"   [LLM {label}; retrying in {wait:.1f}s ({attempt}/{max_retries})]{suffix}")
        time.sleep(wait)
        delay_cap = min(delay_cap * 2, MAX_DELAY_S)

    raise RuntimeError("unreachable")  # the loop above always returns or raises


def call_llm(
    system_prompt: str,
    user_prompt: str,
    model: str,
    response_schema: dict | None = None,
    temperature: float | None = None,
    reasoning_effort: str | None = None,
) -> dict:
    """Single-key call_llm() -- rebound to _call_llm_with_rotation by the key-pool cell below,
    which every downstream cell in this notebook calls by this same name."""
    return _call_llm_single_key(
        LLM_API_KEY,
        system_prompt,
        user_prompt,
        model,
        response_schema,
        temperature,
        reasoning_effort,
        MAX_LLM_RETRIES,
    )

### ## 2.1 Key-pool rotation (shared with `03_evaluation.ipynb`)

Both Groq's and Gemini's free tiers cap a single key's request volume enough that iterating on this notebook can burn through a day's quota fast. Reads `GROQ_API_KEY` + `GROQ_API_KEY_1`..`GROQ_API_KEY_21` when
`LLM_PROVIDER == "groq"`, or `LLM_API_KEY` + `LLM_API_KEY1`..`LLM_API_KEY11` when
`LLM_PROVIDER == "gemini"` (same naming each provider's own notebook already used before this
merge -- kept as-is rather than unified, since `.env` files already in use follow these exact
names). Rebinds the global `call_llm` name to a rotation-aware wrapper -- every existing call
site (`understand_query()`, `answer()`) already calls `call_llm(...)` by name,
so nothing else in this notebook needs to change.

Each pool key gets exactly one fast attempt (`MAX_LLM_RETRIES` temporarily dropped to 1)
before rotating -- a 429 is rejected before any generation happens, so it costs no real
tokens, and cycling through all configured keys takes seconds. Only once every key has failed
once does it fall back to one full retry/backoff pass (`MAX_LLM_RETRIES` restored) on the
current key, in case the failure was actually transient rather than the whole pool being
genuinely exhausted.

**Rate-limit awareness, added 2026-09-18:** before trying a key, `_call_llm_with_rotation()`
checks it against `RATE_LIMITS[LLM_PROVIDER]` (Groq: 30 RPM / 1000 RPD; Gemini: 15 RPM / 1500
RPD, both confirmed free-tier figures) via `_key_available()` -- a key already at its own local
limit is skipped with **no HTTP request made**, not tried and left to 429. If every pool key is
at its limit, `call_llm()` raises `AllKeysRateLimitedError` without sending anything at all.
This is a local, approximate, kernel-session-local guard on top of (not instead of) the
provider's own reactive 429 handling above.

In [ ]:
if LLM_PROVIDER == "gemini":
    _KEY_POOL = [LLM_API_KEY] if LLM_API_KEY else []
    for _i in range(1, 12):
        _key = os.environ.get(f"LLM_API_KEY{_i}", "").strip()
        if _key:
            _KEY_POOL.append(_key)
else:
    _KEY_POOL = [LLM_API_KEY] if LLM_API_KEY else []
    for _i in range(1, 22):
        _key = os.environ.get(f"GROQ_API_KEY_{_i}", "").strip()
        if _key:
            _KEY_POOL.append(_key)
print(f"{len(_KEY_POOL)} {LLM_PROVIDER} key(s) available for rotation")

_key_pool_index = 0
_call_llm_no_rotation = call_llm  # the un-wrapped version, defined in the cell above
_ORIGINAL_MAX_LLM_RETRIES = MAX_LLM_RETRIES

# Per-key fixed-window request counters (RPM + RPD), backing _key_available()/_record_key_usage()
# below -- see RATE_LIMITS (previous cell) for the actual per-provider numbers. Fixed windows,
# not a rolling one -- simpler, and "approximately N requests per minute/day" is the actual
# goal (a client-side safety margin, not exact provider-side parity). Kernel-session-local:
# restarting the kernel resets these, same as _KEY_POOL itself.
_minute_window: dict[str, int] = {}
_minute_count: dict[str, int] = {}
_day_window: dict[str, int] = {}
_day_count: dict[str, int] = {}


def _key_available(key: str) -> bool:
    limits = RATE_LIMITS[LLM_PROVIDER]
    minute = int(time.monotonic() // 60)
    day = int(time.time() // 86400)
    in_minute = _minute_window.get(key) == minute
    in_day = _day_window.get(key) == day
    minute_count = _minute_count.get(key, 0) if in_minute else 0
    day_count = _day_count.get(key, 0) if in_day else 0
    return minute_count < limits["rpm"] and day_count < limits["rpd"]


def _record_key_usage(key: str) -> None:
    minute = int(time.monotonic() // 60)
    day = int(time.time() // 86400)
    if _minute_window.get(key) != minute:
        _minute_window[key] = minute
        _minute_count[key] = 0
    _minute_count[key] += 1
    if _day_window.get(key) != day:
        _day_window[key] = day
        _day_count[key] = 0
    _day_count[key] += 1


def _call_llm_with_rotation(
    system_prompt: str,
    user_prompt: str,
    model: str,
    response_schema: dict | None = None,
    temperature: float | None = None,
    reasoning_effort: str | None = None,
) -> dict:
    """Rotation-aware call_llm(). Before ever calling out, each candidate key is checked
    against its own local RPM/RPD budget (_key_available()) -- a key already at its limit is
    skipped with no HTTP request made. If every pool key is at its limit, raises
    AllKeysRateLimitedError without sending anything.

    Otherwise tries each available key with exactly one fast attempt (MAX_LLM_RETRIES
    temporarily set to 1, so a 429 raises immediately instead of honoring the server's
    Retry-After) before moving to the next key -- see this section's markdown for why the
    naive "let each key retry fully, then rotate" version was too slow to use. Falls through
    to the plain single-key behavior (full retry/backoff) when _KEY_POOL is empty, and as a
    last resort after every pool key has failed once or is rate-limited, in case the failure
    was actually transient rather than the whole pool being genuinely exhausted."""
    global LLM_API_KEY, _key_pool_index, MAX_LLM_RETRIES
    if not _KEY_POOL:
        return _call_llm_no_rotation(
            system_prompt, user_prompt, model, response_schema, temperature, reasoning_effort
        )
    total_keys = len(_KEY_POOL)
    MAX_LLM_RETRIES = 1
    try:
        for attempt in range(total_keys):
            key = _KEY_POOL[_key_pool_index]
            if not _key_available(key):
                print(
                    f"   [key #{_key_pool_index + 1}/{total_keys} at its local rate limit -- "
                    "skipping, no call made]"
                )
                _key_pool_index = (_key_pool_index + 1) % total_keys
                continue
            _record_key_usage(key)
            LLM_API_KEY = key
            try:
                return _call_llm_single_key(
                    key,
                    system_prompt,
                    user_prompt,
                    model,
                    response_schema,
                    temperature,
                    reasoning_effort,
                    MAX_LLM_RETRIES,
                )
            except urllib.error.HTTPError:
                if attempt < total_keys - 1:
                    failed_key = _key_pool_index + 1
                    _key_pool_index = (_key_pool_index + 1) % total_keys
                    print(
                        f"   [key #{failed_key}/{total_keys} failed fast -- rotating to key "
                        f"#{_key_pool_index + 1}/{total_keys}]"
                    )
    finally:
        MAX_LLM_RETRIES = _ORIGINAL_MAX_LLM_RETRIES

    fallback_key = _KEY_POOL[_key_pool_index]
    if not _key_available(fallback_key):
        fallback_key = next((k for k in _KEY_POOL if _key_available(k)), None)
    if fallback_key is None:
        raise AllKeysRateLimitedError(
            f"All {total_keys} pool key(s) are at their local {LLM_PROVIDER} rate limit "
            f"({RATE_LIMITS[LLM_PROVIDER]['rpm']} RPM / {RATE_LIMITS[LLM_PROVIDER]['rpd']} RPD) "
            "-- not sending this request."
        )
    _record_key_usage(fallback_key)
    LLM_API_KEY = fallback_key
    print(
        "   [every pool key failed once or was rate-limited -- falling back to full retry/backoff]"
    )
    return _call_llm_single_key(
        fallback_key,
        system_prompt,
        user_prompt,
        model,
        response_schema,
        temperature,
        reasoning_effort,
        MAX_LLM_RETRIES,
    )


call_llm = _call_llm_with_rotation  # every call site below already calls call_llm() by name

## 3. Query understanding (LLM)

Replaces two heuristics with one LLM call: `01`'s regex `parse_constraints()` (dietary/price
wording, allergy wording) and this notebook's earlier `classify_intent()` (guessing intent
from whichever row retrieval happened to rank first). `understand_query()` asks the model
directly, before any retrieval happens, for:

- `intent` -- `menu` or `faq`.
- `dietary` -- `vegan`/`vegetarian`/`none`. `none` on purpose for a general availability
  question ("do you have vegan options") -- `01` learned that hard-filtering those loses FAQ
  rows, since FAQ rows carry no `dietary_tags` of their own.
- `price_max_gbp` -- only for a firm ceiling ("under £8"), not vague wording ("affordable").
- `allergens_exclude` -- canonical allergen names, constrained by `response_schema`'s `enum`
  to the same vocabulary `01` used for its regex synonym map, so the model can't return a
  value the corpus wouldn't recognise.
- `search_query` -- the question with dietary/price/allergy wording stripped out, since that's
  already handled by `dietary`/`price_max_gbp`/`allergens_exclude` above. Retrieval and rerank
  use this instead of the raw question (section 4's `search()`), so "vegan" and "under £6"
  stop diluting the vector match for the word that actually identifies the dish.
- `category_hint` -- zero or more of the menu's own category names, guessed from course-type
  language ("starter", "main", "dessert", "drink"). Constrained by `response_schema`'s `enum`
  to `MENU_CATEGORIES`, fetched live from the collection below rather than a hardcoded
  synonym list, so it can't drift from the real menu. `expand_category_hint()` then adds
  sibling categories -- ones sharing an immediate parent in `category_path` -- so getting one
  sibling right pulls in the rest without expecting the model to enumerate every one from
  memory. Folded into the search text as a soft signal (section 4), not a hard filter -- a
  wrong guess should never be able to hide the right dish, only fail to help find it.
- `gluten_free_only` -- true ONLY when the guest asks for the restaurant's own curated
  gluten-free menu/section by name ("what's on your gluten-free menu", "gluten-free options").
  This is a positive filter on `is_gluten_free_listed`, separate from `allergens_exclude`: a
  guest describing an actual allergy ("I'm coeliac", "no gluten please") should still get
  `allergens_exclude` populated too (the safety exclusion), but a guest just browsing the
  named section doesn't need every gluten-containing dish removed from consideration first.
- `kcal_max` -- a number for a calorie ceiling, whether firm ("under 500 calories") or
  qualitative ("a low-calorie main" -> use a sensible reference like 500). null otherwise.
- `protein_min_g` -- a number for a protein floor, whether firm ("at least 20g protein") or
  qualitative ("a high-protein dish" -> use a sensible reference like 20). null otherwise.
  Unlike `price_max_gbp`, qualitative wording is honoured here with a reasonable default
  rather than left null, since "high protein" carries real meaning a guest expects acted on,
  the way "affordable" doesn't for price.
- `alcohol_free` -- true ONLY when the guest explicitly wants a non-alcoholic / alcohol-free
  drink. A positive filter on `abv_percent` being unset.

A live worked example: **"a vegan starter under £6" returned NO CONFIDENT MATCH** the first
time this notebook ran that question, even though the corpus does have exactly one vegan
starter under £6 (an oyster + shiitake mushroom bao bun) -- `data/knowledge_base.json` has no
category literally called "starters", so the raw question's vector match against 39 other
vegan-and-cheap-but-irrelevant rows (mostly drinks and desserts) drowned out the one real
match. Adding `search_query` + `category_hint` alone wasn't enough on the next run either:
the model guessed `lighter bites` and `big flavour bites` but not `bao buns` -- the category
the real match is actually in. All four (`bao buns`, `big flavour bites`, `gyoza`,
`lighter bites`) turn out to share the same `category_path` parent, `sides` -- this menu's
own closest thing to a starters section, just never labelled that anywhere. (Confusingly,
`sides` is *also* used as its own unrelated leaf category, but only under `gluten free`'s
menu section -- nothing to do with this group; a corpus vocabulary quirk in its own right.)
`expand_category_hint()` reads that sibling relationship straight from `category_path`, so
the two correct guesses now pull `bao buns` in too.

This call uses `UNDERSTAND_TEMPERATURE = 0.0` (deterministic) plus Groq's strict
`json_schema` response format. The schema does the heavy lifting regardless of
temperature -- it constrains the output's structure and every enum-typed field so the
model literally cannot return an invalid category or allergen name -- and
`temperature=0` additionally makes the free-form fields (`search_query`,
`price_max_gbp`, ...) as repeatable as this kind of extraction gets.

In [ ]:
NUT_ALLERGENS = {
    "peanuts",
    "tree nuts",
    "almond nuts",
    "walnuts",
    "hazelnuts",
    "pecan nuts",
    "pistachios",
    "brazil nuts",
    "cashew nuts",
    "macadamia nuts",
}
ALLERGEN_VOCAB: dict[str, set[str]] = {
    "peanut": {"peanuts"},
    "nut": NUT_ALLERGENS,
    "gluten": {"cereals containing gluten", "wheat", "barley", "oats", "rye"},
    "wheat": {"wheat", "cereals containing gluten"},
    "dairy": {"milk"},
    "milk": {"milk"},
    "egg": {"eggs"},
    "soy": {"soya"},
    "soya": {"soya"},
    "sesame": {"sesame"},
    "shellfish": {"crustaceans", "molluscs"},
    "crustacean": {"crustaceans"},
    "fish": {"fish"},
    "celery": {"celery"},
    "mustard": {"mustard"},
    "sulphite": {"sulphites"},
    "lupin": {"lupin"},
}
ALLOWED_ALLERGENS = sorted(set().union(*ALLERGEN_VOCAB.values()))

_cat_props = ["category", "category_path", "item_type"]
MENU_CATEGORIES_SET: set[str] = set()
_parent_groups: dict[str, set[str]] = {}
for o in kb.iterator(return_properties=_cat_props):
    props = o.properties
    if props.get("item_type") != "menu_item":
        continue
    category = props.get("category")
    if not isinstance(category, str):
        continue
    MENU_CATEGORIES_SET.add(category)
    path = props.get("category_path")
    if isinstance(path, list) and len(path) >= 2 and isinstance(path[0], str):
        _parent_groups.setdefault(path[0], set()).add(category)
MENU_CATEGORIES = sorted(MENU_CATEGORIES_SET)

# Sibling categories sharing an immediate parent in category_path (e.g. bao buns / gyoza /
# lighter bites / big flavour bites are all "sides > X" -- this menu's own closest thing to a
# starters section, never labelled that anywhere). If the model gets one sibling right, this
# expands category_hint to the rest -- more reliable than expecting it to enumerate every
# sibling from memory, and it can't drift since it's read from the corpus's own structure.
# Sibling expansion is a course-type feature (bao buns / gyoza / lighter bites / big flavour
# bites are all genuinely interchangeable "starter"-type dishes under one parent) -- excluded
# here for "drinks" specifically, since its sub-categories (coffee + tea, wine + sake,
# beers + cider, cocktails, soft drinks, freshly made juices) are mutually exclusive drink
# TYPES, not synonyms. Found live: "is there coffee?" expanded category_hint to all six
# adult-beverage categories, diluting both retrieval and the rerank query text badly enough
# that every genuine coffee/tea row lost to unrelated wine/juice/cider rows that merely
# happened to have richer description text.
CATEGORY_SIBLINGS: dict[str, set[str]] = {}
for _parent, _siblings in _parent_groups.items():
    if _parent == "drinks":
        continue
    for _leaf in _siblings:
        CATEGORY_SIBLINGS[_leaf] = _siblings


def expand_category_hint(category_hint: list[str]) -> list[str]:
    """Add sibling categories (same category_path parent) to a guessed category_hint."""
    expanded = set(category_hint)
    for c in category_hint:
        expanded |= CATEGORY_SIBLINGS.get(c, set())
    return sorted(expanded)


# The categories under the "drinks" parent that are never alcohol-free -- read from the
# corpus's own structure the same way CATEGORY_SIBLINGS is, so it can't drift if a category
# is renamed. Used to keep alcohol_free's soft category_hint text from fighting its own filter.
ALCOHOLIC_ONLY_CATEGORIES = _parent_groups.get("drinks", set()) - {
    "coffee + tea",
    "soft drinks",
    "freshly made juices",
}


UNDERSTAND_SYSTEM_PROMPT = f"""
You turn one guest question for a restaurant chatbot into a structured query-understanding
result used to search a knowledge base. Return only the JSON described by the response
schema -- no extra text.

Fields:
- intent: "menu" if the guest is asking about a dish, ingredient, price, or nutrition value --
  this includes comparing two or more named dishes to each other (e.g. "what's the difference
  between the yasai cha han and the vegan recipe version" is still a menu question, not FAQ,
  even though it doesn't ask about just one dish), and includes a general browse/availability
  question about what's on the menu (e.g. "do you have vegan options", "what desserts do you
  have") even when no specific dish is named. "faq" if they're asking about restaurant policy
  itself -- hours, bookings, delivery, payments, gift cards, or where to find allergen
  information -- not about menu content.
- dietary: "vegan" or "vegetarian" ONLY when the guest wants dishes filtered to that
  restriction (e.g. "a vegan curry", "vegetarian mains"). Use "none" for a general
  availability question like "do you have vegan options" -- that should still search
  everything rather than be filtered down, since FAQ rows about dietary options carry no
  dietary_tags of their own and a hard filter would hide them.
- price_max_gbp: a number ONLY when the guest gives a firm ceiling ("under £8", "less than
  £10"). null for vague wording like "affordable" or "cheap".
- allergens_exclude: canonical allergen names (from the list below) the guest wants excluded,
  ONLY when they state an allergy, intolerance, or something to avoid (e.g. "I have a nut
  allergy", "dairy-free options", "no shellfish"). Map colloquial terms to every matching
  canonical value -- "nuts" maps to every tree-nut entry plus peanuts, "dairy" maps to milk,
  "shellfish" maps to crustaceans and molluscs. Empty array if no allergy was stated.
- search_query: the question rewritten as a short search phrase for just the dish/food itself
  -- strip out anything already captured by dietary, price_max_gbp, or allergens_exclude
  above (don't repeat "vegan", "under £6", or allergy wording) and strip filler words ("a",
  "do you have", "what's in"). Examples: "a vegan starter under £6" -> "starter";
  "a spicy noodle dish under £8" -> "spicy noodle dish"; "what time do you open" -> "what
  time do you open" (nothing to strip for an FAQ question). EXCEPTION: a "(gluten-free
  recipe)" or "(vegan recipe)" suffix is part of that dish's own name on this menu -- two
  different recipes can share the same display name, disambiguated only by this suffix -- so
  if the guest names a dish that way, KEEP the suffix verbatim in search_query even though it
  reads like dietary wording. Example: "I'm vegan, is the yasai cha han (vegan recipe) safe"
  -> search_query "yasai cha han (vegan recipe)", NOT "yasai cha han" (stripping it searches
  for a different recipe with different allergens than the one actually asked about). Never
  return an empty string -- fall back to the original question if nothing else to extract.
- category_hint: zero or more names from the menu category list below that best match any
  course-type language in the question (e.g. "starter", "small plate", "main", "dessert",
  "drink") -- the guest's word for a course type rarely matches this menu's own category
  names exactly (there is no category literally called "starters"), so use your judgement
  about which real categories a guest asking for that course type would actually mean.
  Leave empty if the question already names a specific dish, or names no course type, or is
  an "faq" question (this list is menu categories only). EXCEPTION: the category literally
  named "drinks" is the kids' menu's drinks section specifically, not general beverages --
  for a guest asking about a drink without saying "kids", use the actual adult beverage
  categories instead ("coffee + tea", "soft drinks", "freshly made juices", "beers + cider",
  "wine + sake", "cocktails"), picking whichever most closely matches what they asked for.
- gluten_free_only: true ONLY when the guest asks for the restaurant's own gluten-free
  menu/section by name ("what's on your gluten-free menu", "gluten-free options"). This is a
  positive filter for that curated section -- separate from allergens_exclude, which is the
  safety exclusion for a guest describing an actual allergy or intolerance. Both can be true
  together (e.g. "I'm coeliac, what's on the gluten-free menu").
- kcal_max: a calorie ceiling as a number. Honour both a firm number ("under 500 calories")
  and qualitative wording ("a low-calorie main" -> use a sensible reference like 500). null
  if calories were not mentioned at all.
- protein_min_g: a protein floor in grams as a number. Honour both a firm number ("at least
  20g protein") and qualitative wording ("a high-protein dish" -> use a sensible reference
  like 20). null if protein was not mentioned at all.
- alcohol_free: true ONLY when the guest explicitly wants a non-alcoholic / alcohol-free
  drink.

Canonical allergens: {", ".join(ALLOWED_ALLERGENS)}
Menu categories: {", ".join(MENU_CATEGORIES)}
""".strip()

# Standard JSON Schema (Groq's strict json_schema mode), not Gemini's responseSchema dialect --
# lowercase types, nullable as a ["type", "null"] union, and additionalProperties: False on
# every object level. Strict mode requires every property to be listed in "required" (a
# property can still resolve to null via its own type union) and forbids extra keys.
UNDERSTAND_TEMPERATURE = 0.0  # deterministic extraction; the schema already constrains shape/enums
UNDERSTAND_REASONING_EFFORT = (
    "low"  # confirmed: same extracted JSON as default, ~44% fewer completion tokens
)
RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "intent": {"type": "string", "enum": ["menu", "faq"]},
        "dietary": {"type": "string", "enum": ["vegan", "vegetarian", "none"]},
        "price_max_gbp": {"type": ["number", "null"]},
        "allergens_exclude": {
            "type": "array",
            "items": {"type": "string", "enum": ALLOWED_ALLERGENS},
        },
        "search_query": {"type": "string"},
        "category_hint": {
            "type": "array",
            "items": {"type": "string", "enum": MENU_CATEGORIES},
        },
        "gluten_free_only": {"type": "boolean"},
        "kcal_max": {"type": ["number", "null"]},
        "protein_min_g": {"type": ["number", "null"]},
        "alcohol_free": {"type": "boolean"},
    },
    "required": [
        "intent",
        "dietary",
        "price_max_gbp",
        "allergens_exclude",
        "search_query",
        "category_hint",
        "gluten_free_only",
        "kcal_max",
        "protein_min_g",
        "alcohol_free",
    ],
    "additionalProperties": False,
}


def understand_query(question: str) -> dict:
    """Single LLM call: classify intent and extract retrieval filters from the question."""
    if not LLM_API_KEY:
        return {
            "intent": "menu",
            "dietary": None,
            "price_max_gbp": None,
            "allergens_exclude": [],
            "search_query": question,
            "category_hint": [],
            "gluten_free_only": False,
            "kcal_max": None,
            "protein_min_g": None,
            "alcohol_free": False,
            "usage": _zero_usage(),
        }
    resp = call_llm(
        UNDERSTAND_SYSTEM_PROMPT,
        question,
        model=UNDERSTAND_MODEL,
        response_schema=RESPONSE_SCHEMA,
        temperature=UNDERSTAND_TEMPERATURE,
        reasoning_effort=UNDERSTAND_REASONING_EFFORT,
    )
    parsed = json.loads(resp["text"])
    dietary = parsed.get("dietary") or "none"
    allergens = [a for a in parsed.get("allergens_exclude", []) if a in ALLOWED_ALLERGENS]
    alcohol_free = bool(parsed.get("alcohol_free"))
    category_hint = [c for c in parsed.get("category_hint", []) if c in MENU_CATEGORIES]
    category_hint = expand_category_hint(category_hint)
    if alcohol_free:
        # Sibling expansion (above) can legitimately pull in every beverage category,
        # alcoholic ones included -- appending those category names as search text would
        # dilute the vector match toward beer/wine/cocktail rows the hard filter is about to
        # exclude anyway, which can drag every genuinely alcohol-free candidate's score below
        # the answerability gate. Drop them from the soft-signal text; the filter alone
        # already handles exclusion correctly.
        category_hint = [c for c in category_hint if c not in ALCOHOLIC_ONLY_CATEGORIES]
    search_query = (parsed.get("search_query") or "").strip() or question
    return {
        "intent": parsed.get("intent") if parsed.get("intent") in ("menu", "faq") else "menu",
        "dietary": None if dietary == "none" else dietary,
        "price_max_gbp": parsed.get("price_max_gbp"),
        "allergens_exclude": allergens,
        "search_query": search_query,
        "category_hint": category_hint,
        "gluten_free_only": bool(parsed.get("gluten_free_only")),
        "kcal_max": parsed.get("kcal_max"),
        "protein_min_g": parsed.get("protein_min_g"),
        "alcohol_free": alcohol_free,
        "usage": resp["usage"],
    }

## 4. Retrieval pipeline

The rest of `01`'s pipeline, unchanged in substance -- only `parse_constraints()` is gone,
replaced by two functions consuming `understand_query()`'s output: `build_filter()` (dietary +
price -> a Weaviate server-side filter, as before) and `build_search_text()` (new -- see
below). `FIELDS` still carries the two additions generation needs: `description` (the actual
grounding text, not just the vectorized `embedding_text`) and `kcal`.

`search()` now retrieves and reranks against `build_search_text(u)`, not the raw question --
`understand_query()`'s cleaned `search_query` plus its `category_hint` appended as plain
text. `search()`'s result dict carries `search_text` too, so it's visible in every trace
alongside `understanding`, not just used silently.

`rerank()` now also returns Cohere's own `meta.billed_units.search_units` for its call --
real billing data (one search unit per up to 100 documents), not an estimate, zero on the
hybrid-order fallback since nothing billable happened. This is the only embedding-side cost
this notebook can actually observe: the hybrid query's own vectorization of your question text
happens *inside* Weaviate's managed `text2vec_cohere` integration, and Weaviate's query
response never reports what that internal Cohere call cost -- there is no client-visible
number for it with this architecture. `show_answer()` (section 8) reports the rerank units
per question and running-session total on that basis, not a full embedding cost.

`rerank()` now paces itself under `COHERE_MAX_RPM` too, the same proactive-pacing idea section
2 already applies to the LLM side (Groq) -- previously only the LLM side had this, but
constraint relaxation below means one question can now trigger several `rerank()` calls
instead of exactly one, which raises the odds of a Cohere 429 the same way multiple LLM
calls did before `_pace_llm_call()` existed.

**Constraint relaxation** -- borrowed from a similar RAG assignment
(`C1M5_Assignment_Solve.ipynb`, which relaxes clothing-store filters like colour and category
when a search comes up too thin). If nothing clears `GATE` on the first attempt, `search()`
drops one `RELAXABLE_FIELDS` constraint at a time (`kcal_max`, `protein_min_g`,
`gluten_free_only`, `alcohol_free`, then `price_max_gbp` last) and retries, instead of
declining outright when a slightly-off-spec match exists. Adapted for a restaurant rather than
copied outright: `dietary` and `allergens_exclude` are **never** in `RELAXABLE_FIELDS` and
never get cleared -- the clothing example relaxes every filter including gender and category,
which is fine for "no exact colour match" but would be actively harmful here (silently
suggesting a non-vegan dish to a vegan guest, or one containing a stated allergen). The
returned `relaxed_fields` list feeds into section 7's `build_user_prompt()`, so the model is
told explicitly which constraint was dropped and states the dish's real figure instead of
implying a false match. `rerank_search_units` accumulates across every relaxation attempt,
not just the last one, since each retry is a genuine extra Cohere call.

In [ ]:
ALPHA = 0.75
K = 20
TOP_N = 6
GATE = 0.15
RERANK_MODEL = "rerank-v3.5"
COHERE_MAX_RPM = 15.0  # client-side pacing cap for rerank() -- edit to match your key's quota

_last_cohere_call_at = 0.0


def _pace_cohere_call() -> None:
    """Block just long enough to keep rerank() under COHERE_MAX_RPM requests/minute."""
    global _last_cohere_call_at
    min_interval = 60.0 / COHERE_MAX_RPM
    wait = min_interval - (time.monotonic() - _last_cohere_call_at)
    if wait > 0:
        time.sleep(wait)
    _last_cohere_call_at = time.monotonic()


FIELDS = [
    "name",
    "category",
    "item_type",
    "description",
    "ingredients",
    "price_gbp",
    "kcal",
    "protein_g",
    "abv_percent",
    "dietary_tags",
    "allergens_contains",
    "allergens_may_contain",
    "is_gluten_free_listed",
    "embedding_text",
]


def pstr(o, key: str) -> str:
    """properties[key] as a string, or "" if it is not one."""
    v = o.properties.get(key)
    return v if isinstance(v, str) else ""


def plist(o, key: str) -> list:
    """properties[key] as a list, or [] if it is not one."""
    v = o.properties.get(key)
    return v if isinstance(v, list) else []


def pnum(o, key: str) -> float | None:
    """properties[key] as a float, or None if it is not numeric."""
    v = o.properties.get(key)
    return float(v) if isinstance(v, (int, float)) else None


def allergen_set(o) -> set[str]:
    """Union of an object's allergens_contains and allergens_may_contain."""
    return {*plist(o, "allergens_contains"), *plist(o, "allergens_may_contain")}


def build_filter(u: dict) -> FilterReturn | None:
    """Turn an understand_query() result into a Weaviate server-side filter."""
    clauses: list[FilterReturn] = []
    if u["dietary"]:
        clauses.append(Filter.by_property("dietary_tags").contains_any([u["dietary"]]))
    if u["price_max_gbp"] is not None:
        clauses.append(Filter.by_property("price_gbp").less_or_equal(float(u["price_max_gbp"])))
    if u["gluten_free_only"]:
        clauses.append(Filter.by_property("is_gluten_free_listed").equal(True))
    if u["kcal_max"] is not None:
        clauses.append(Filter.by_property("kcal").less_or_equal(float(u["kcal_max"])))
    if u["protein_min_g"] is not None:
        clauses.append(Filter.by_property("protein_g").greater_or_equal(float(u["protein_min_g"])))
    if u["alcohol_free"]:
        # 0.5% ABV is the standard UK low/no-alcohol labeling threshold. abv_percent is
        # populated on every drink row (0.0 for confirmed non-alcoholic, real ABV otherwise)
        # and left null on food rows -- a range filter naturally excludes null, so this
        # never needs an explicit IS NULL check.
        clauses.append(Filter.by_property("abv_percent").less_or_equal(0.5))
    return Filter.all_of(clauses) if clauses else None


def build_search_text(u: dict) -> str:
    """Turn an understand_query() result into the text used for hybrid retrieval + rerank.

    Uses search_query (dietary/price/allergy wording already stripped) instead of the raw
    question, so those tokens stop diluting the vector match once build_filter() has already
    handled them deterministically. category_hint is appended as a soft signal -- it's plain
    text, not a hard filter, so a wrong guess can only fail to help, never hide the right row.
    """
    text = u["search_query"]
    if u["category_hint"]:
        text = f"{text} ({', '.join(u['category_hint'])})"
    return text


def retrieve(query: str, k: int = K, alpha: float = ALPHA, filters: FilterReturn | None = None):
    """Run one hybrid (keyword + vector) query against KnowledgeBase."""
    return kb.query.hybrid(
        query=query,
        alpha=alpha,
        limit=k,
        filters=filters,
        return_properties=FIELDS,
        return_metadata=MetadataQuery(score=True),
    ).objects


def rerank(query: str, objs: list, top_n: int = TOP_N) -> tuple[list[dict], int]:
    """Rerank objs against query with Cohere; fall back to hybrid order on rate limit.

    Paces itself under COHERE_MAX_RPM before every attempt -- constraint relaxation in
    search() can call this more than once per question, so this needs the same proactive
    pacing call_llm() has, not just the existing reactive 429 retry below. Returns
    (ranked_hits, search_units) -- search_units is Cohere's own meta.billed_units.search_units
    for this call (real billing data, not an estimate); 0 on the hybrid-order fallback, since
    no billable rerank call actually completed.
    """
    if not objs:
        return [], 0
    docs = [pstr(o, "embedding_text") or pstr(o, "name") for o in objs]
    payload = json.dumps(
        {"model": RERANK_MODEL, "query": query, "documents": docs, "top_n": min(top_n, len(docs))}
    ).encode()
    results = None
    search_units = 0
    for attempt in range(4):
        _pace_cohere_call()
        try:
            req = urllib.request.Request(
                "https://api.cohere.com/v2/rerank",
                data=payload,
                headers={
                    "Authorization": f"Bearer {COHERE_KEY.strip()}",
                    "Content-Type": "application/json",
                },
                method="POST",
            )
            with urllib.request.urlopen(req, timeout=30) as resp:
                data = json.load(resp)
            results = data["results"]
            search_units = data.get("meta", {}).get("billed_units", {}).get("search_units", 0)
            break
        except urllib.error.HTTPError as e:
            retryable = e.code == 429 and attempt < 3
            code_ = e.code
            e.close()
            if retryable:
                time.sleep(2 * 4**attempt)
            else:
                print(f"   [rerank HTTP {code_}; using hybrid order]")
                break
    if results is None:
        hits = [
            {"obj": o, "rerank": math.nan, "hybrid": o.metadata.score or 0.0} for o in objs[:top_n]
        ]
        return hits, 0
    hits = [
        {
            "obj": objs[r["index"]],
            "rerank": float(r["relevance_score"]),
            "hybrid": objs[r["index"]].metadata.score or 0.0,
        }
        for r in results
    ]
    return hits, search_units


RELAXABLE_FIELDS = [
    "kcal_max",
    "protein_min_g",
    "gluten_free_only",
    "alcohol_free",
    "price_max_gbp",
]


def _is_constraint_set(u: dict, field: str) -> bool:
    return bool(u[field]) if field in ("gluten_free_only", "alcohol_free") else u[field] is not None


def _clear_constraint(u: dict, field: str) -> dict:
    """A copy of u with one relaxable constraint cleared back to its unset value."""
    cleared = dict(u)
    cleared[field] = False if field in ("gluten_free_only", "alcohol_free") else None
    return cleared


def search(question: str, gate: float = GATE) -> dict:
    """Full pipeline: understand the query (LLM), retrieve, exclude allergens, rerank, gate.

    If nothing clears the gate, progressively drops one RELAXABLE_FIELDS constraint at a time
    (least essential first) and retries, rather than declining outright when a closer, slightly
    off-spec match exists. dietary and allergens_exclude are never relaxed -- those are safety
    and preference guarantees, not refinements, and silently dropping either could put an unsafe
    or unwanted dish in front of a guest.
    """
    u = understand_query(question)
    current = u
    relaxed_fields: list[str] = []
    total_search_units = 0
    while True:
        server = build_filter(current)
        exclude = set(current["allergens_exclude"])
        search_text = build_search_text(current)
        objs = retrieve(search_text, filters=server)
        kept = [o for o in objs if not (allergen_set(o) & exclude)] if exclude else objs
        ranked, search_units = rerank(search_text, kept)
        total_search_units += search_units
        top = ranked[0]["rerank"] if ranked else 0.0
        answerable = bool(ranked) if math.isnan(top) else top >= gate
        if answerable:
            break
        next_field = next((f for f in RELAXABLE_FIELDS if _is_constraint_set(current, f)), None)
        if next_field is None:
            break
        relaxed_fields.append(next_field)
        current = _clear_constraint(current, next_field)

    # A guest naming a specific dish is most likely asking about the single best name-match
    # in the WHOLE corpus, filters aside. If a dietary hard-filter (build_filter()) or the
    # allergen exclude just kept that exact dish out of `ranked` entirely, that has to reach
    # generation explicitly -- otherwise nothing distinguishes "the dish you asked about
    # doesn't meet your requirement" from "here's a different dish that happens to rank high",
    # and the model can silently conflate the two. One extra unfiltered lookup catches both
    # exclusion paths (found live: allergens_exclude conflated two chicken-katsu dishes;
    # dietary="vegan" conflated a "(vegan recipe)" variant with its actually-vegan base dish,
    # since build_filter()'s dietary_tags filter runs server-side before objs is ever built).
    excluded_top_match = None
    if current["dietary"] or exclude:
        unfiltered = retrieve(search_text, k=1, filters=None)
        if unfiltered:
            top_obj = unfiltered[0]
            kept_names = {pstr(h["obj"], "name") for h in ranked}
            if pstr(top_obj, "name") not in kept_names:
                reasons = []
                hit_allergens = allergen_set(top_obj) & exclude
                if hit_allergens:
                    reasons.append(f"contains {', '.join(sorted(hit_allergens))}")
                if current["dietary"] and current["dietary"] not in plist(top_obj, "dietary_tags"):
                    reasons.append(f"is not tagged {current['dietary']}")
                if reasons:
                    excluded_top_match = {
                        "name": pstr(top_obj, "name"),
                        "reason": "; ".join(reasons),
                    }

    # retrieved_objects/kept_objects carry the actual Weaviate hits (not just counts) purely
    # for section 9's explain_pipeline() to render a before/after rerank comparison -- nothing
    # downstream of search() (build_context, generation) reads either field.
    return {
        "understanding": u,
        "relaxed_fields": relaxed_fields,
        "excluded_top_match": excluded_top_match,
        "search_text": search_text,
        "excluded": sorted(exclude),
        "retrieved": len(objs),
        "retrieved_objects": objs,
        "kept": len(kept),
        "kept_objects": kept,
        "ranked": ranked,
        "top": top,
        "answerable": answerable,
        "rerank_search_units": total_search_units,
    }


def line(o) -> str:
    """One-line summary of an object: type, name, category, price, dietary tags."""
    price = pnum(o, "price_gbp")
    price_s = f"  £{price:.2f}" if price is not None else ""
    diet = plist(o, "dietary_tags")
    diet_s = f"  {diet}" if diet else ""
    return f"[{pstr(o, 'item_type')}] {pstr(o, 'name')} <{pstr(o, 'category')}>{price_s}{diet_s}"

## 5. Tone policy

Two levers now, working together, both picked from the understanding step's intent: a tone
instruction (what to say) and a real `temperature` (how much to vary phrasing). This section
used to be "Temperature policy" (`TEMP_MENU = 0.2` / `TEMP_FAQ = 0.8`) before this notebook's
Gemini backend, then became tone-only once `gemini-3.8-flash` turned out to silently ignore
`temperature`/`top_p`/`top_k` entirely (confirmed against Google's own docs and forum). Groq's
`openai/gpt-oss-120b` genuinely applies `temperature` -- confirmed directly in this notebook's
Phase 1 smoke test (`temperature=0` returned the identical sentence three times in a row;
`temperature=1.8` varied every time) -- so both levers are back, and they control different
things: the instruction shapes *content*, the temperature shapes *phrasing variability*.

- **Menu** -- precise and literal tone, `temperature=0.2`. Price, allergens, and nutrition are
  facts pulled from `CONTEXT`; the instruction tells the model to stay close to `CONTEXT`'s
  exact wording rather than paraphrase a number or an allergen into something wrong, and the
  low temperature keeps phrasing close to the same safe wording call to call.
- **FAQ** -- warm and conversational tone, `temperature=0.8`. House-policy answers (hours,
  bookings, payments) are copy, not numbers; the instruction gives it room to phrase things
  naturally as long as the substance still matches `CONTEXT`, and the higher temperature lets
  that natural phrasing actually vary.

In [ ]:
MENU_TONE = (
    "Tone for this answer: precise and literal. This is a factual menu question -- stick "
    "closely to CONTEXT's exact wording for prices, allergens, dietary tags, and nutrition "
    "figures. Do not paraphrase or round a number, and do not add warmth or small talk that "
    "risks softening a factual claim."
)
FAQ_TONE = (
    "Tone for this answer: warm and conversational. This is a house-policy question -- feel "
    "free to phrase the answer naturally, in your own words, as long as the substance matches "
    "CONTEXT exactly."
)
MENU_TEMPERATURE = 0.2
FAQ_TEMPERATURE = 0.8
# Fewer reasoning tokens before the answer starts -- the answer is a rewording of CONTEXT
# rather than open-ended reasoning. Mirrors app/agent/generation.py. Unlike the understand
# step's setting, NOT yet checked against this evaluation's scorecard bar -- re-run
# 03_evaluation.ipynb sections 5-8 before relying on it.
GENERATION_REASONING_EFFORT = "low"


def tone_for(intent: str) -> str:
    return FAQ_TONE if intent == "faq" else MENU_TONE


def temperature_for(intent: str) -> float:
    return FAQ_TEMPERATURE if intent == "faq" else MENU_TEMPERATURE

## 6. System instructions (generation)

The grounding rules sent as the model's system instruction on every generation call -- fixed
text, independent of intent (section 8 appends `tone_for(intent)` to it per call), and
distinct from `UNDERSTAND_SYSTEM_PROMPT` above (that one extracts filters; this one answers
the guest). It encodes the project's own
data caveats directly: allergen safety needs the `contains`/`may_contain` union,
and the `(gluten-free recipe)` / `(vegan recipe)` suffix marks a genuinely different dish, not
a duplicate.

In [ ]:
GENERATION_SYSTEM_PROMPT = """
You are the menu assistant for a restaurant chatbot. Answer ONLY using the CONTEXT rows given
with the question below -- they come from the restaurant's own knowledge base. Never use
outside knowledge about food, menus, or any restaurant, and never invent a dish, price, or
policy that is not in CONTEXT.

Rules:
- If CONTEXT is empty or does not answer the question, say plainly that you don't have that
  information and suggest asking a member of staff. Do not guess. EXCEPTION: if a NOTE appears
  below CONTEXT, the NOTE is itself a real, verified answer about a specific dish -- treat it
  exactly like a CONTEXT row, not like missing information. Never say you don't have
  information, and never tell the guest to check with staff instead of answering, when a NOTE
  already tells you the answer -- state the NOTE's fact directly (e.g. why a dish is unsafe or
  doesn't qualify), the same way you would state a fact from a normal CONTEXT row.
- State each dish's price exactly as given in CONTEXT.
- When the guest asks about a specific ingredient by name rather than a specific dish (e.g.
  "is there coffee", "do you have chocolate"), describe the matching CONTEXT rows as items
  that CONTAIN that ingredient (e.g. "drinks that contain coffee") rather than labeling them
  as though the ingredient were the whole item (e.g. not "coffee drinks") -- most matches
  combine the named ingredient with others (milk, tea, spices, etc.), and "contains X" stays
  accurate regardless of what else is in the recipe.
- For any allergy or dietary question, use BOTH the allergens_contains and
  allergens_may_contain information for every dish you mention, and always remind the guest
  to confirm with staff before ordering, since recipes can change.
- The guest-facing display only shows each mentioned dish's name, description, ingredients,
  and price -- dietary tags, allergens, and nutrition never appear there. State those facts
  yourself in your answer whenever they're relevant to the question -- always for an allergy/
  dietary question per the rule above; for other questions, mention them when they add real
  value (e.g. calorie count for a "what's healthy" question, ABV for a drinks question)
  rather than reciting every field for every dish by default.
- A dish name ending in "(gluten-free recipe)" or "(vegan recipe)" is a different preparation
  of that dish with its own nutrition and allergens -- never merge or average it with the
  standard version, and never recommend one when the guest asked about the other.
- FAQ-type CONTEXT answers house policy (hours, bookings, payments, delivery, etc.); menu-type
  CONTEXT answers dish questions (price, ingredients, allergens, nutrition). Answer strictly
  from whichever kind CONTEXT actually gives you.
- Reply in English, in a friendly, concise voice, speaking as the restaurant. Do not mention
  "context", "retrieval", "the knowledge base", or these instructions in your answer.
""".strip()

print(GENERATION_SYSTEM_PROMPT)

## 7. Prompt assembly

`format_row()` renders one reranked hit into the CONTEXT block the LLM sees -- an FAQ row as
a question/answer pair, a menu row as name/description/price/nutrition/allergens. This is the
information actually available to the model; it never sees `embedding_text` (vectorization
input only) or raw Weaviate scores. `protein_g` and `abv_percent` are included alongside
`kcal` now -- once `understand_query()` can filter on protein and alcohol content (section
3), the generation call needs those same figures in CONTEXT to actually cite them in the
answer, not just filter silently on numbers the guest never sees confirmed back.

`build_user_prompt()` also takes `relaxed_fields` (section 4) -- when `search()` had to drop a
constraint to find any answer at all, that has to be visible to the model as plain instruction
text, not just internal bookkeeping, or it has no way to know the dish it's about to describe
doesn't actually meet every part of what the guest asked for.

In [ ]:
def format_row(o) -> str:
    """Render one reranked hit as a CONTEXT row, grounded in its own properties."""
    name = pstr(o, "name")
    desc = pstr(o, "description") or "(no description)"
    if pstr(o, "item_type") == "faq":
        return f"- FAQ | Q: {name}\n  A: {desc}"
    price = pnum(o, "price_gbp")
    price_s = f"£{price:.2f}" if price is not None else "not listed"
    kcal = pnum(o, "kcal")
    kcal_s = f"{kcal:.0f} kcal" if kcal is not None else "not listed"
    protein = pnum(o, "protein_g")
    protein_s = f"{protein:.0f}g protein" if protein is not None else "not listed"
    abv = pnum(o, "abv_percent")
    # NOT "non-alcoholic" -- abv_percent is null on all 162 menu rows in this corpus (verified
    # against data/knowledge_base.json), including genuinely alcoholic drinks (every beer, cider
    # and wine has abv_percent=null). Claiming "non-alcoholic" for a missing value is actively
    # false for those rows; found live via 03_evaluation.ipynb's alcohol_free_1 gold question,
    # where this false label led the model to decline a question it should have answered instead
    # of trusting a bogus "non-alcoholic" claim about an actual lager/wine/sake row.
    abv_s = f"{abv:.1f}% ABV" if abv is not None else "ABV not listed"
    ingredients = ", ".join(plist(o, "ingredients")) or "not listed"
    diet = ", ".join(plist(o, "dietary_tags")) or "none listed"
    contains = ", ".join(plist(o, "allergens_contains")) or "none declared"
    may = ", ".join(plist(o, "allergens_may_contain")) or "none declared"
    gf = "yes" if o.properties.get("is_gluten_free_listed") else "no"
    return (
        f"- MENU ITEM | {name} <{pstr(o, 'category')}>\n"
        f"  description: {desc}\n"
        f"  ingredients: {ingredients}\n"
        f"  price: {price_s}  |  kcal: {kcal_s}  |  protein: {protein_s}  |  {abv_s}  |  "
        f"gluten-free listed: {gf}\n"
        f"  dietary_tags: {diet}\n"
        f"  allergens_contains: {contains}  |  allergens_may_contain: {may}"
    )


def build_context(ranked: list[dict]) -> str:
    """CONTEXT block handed to the LLM: one formatted row per reranked hit."""
    if not ranked:
        return "(no matching rows retrieved)"
    return "\n".join(format_row(h["obj"]) for h in ranked)


def build_user_prompt(
    question: str,
    context: str,
    relaxed_fields: list[str],
    excluded_top_match: dict | None = None,
) -> str:
    """The full user-turn text sent to the LLM alongside GENERATION_SYSTEM_PROMPT.

    When search() had to relax a constraint to find any answerable match, that has to reach
    the model explicitly -- otherwise it has no way to know a shown dish doesn't actually meet
    every part of the original ask, and could misreport it as a full match. Likewise, when the
    single best name-match in the whole corpus was excluded from CONTEXT entirely -- by a
    dietary hard-filter or the allergen exclude -- that has to reach the model explicitly too,
    otherwise nothing stops it from answering as if a different CONTEXT row is the dish the
    guest actually named.
    """
    text = f"QUESTION: {question}\n\nCONTEXT:\n{context}"
    if relaxed_fields:
        text += (
            f"\n\nNOTE: no result matched every part of the question. To surface a closest "
            f"match, these constraints were dropped: {', '.join(relaxed_fields)}. Be upfront "
            f"that the dish doesn't fully satisfy {', '.join(relaxed_fields)} -- state its "
            f"actual figure from CONTEXT rather than implying it meets the original ask."
        )
    if excluded_top_match:
        text += (
            f"\n\nNOTE: '{excluded_top_match['name']}' was the closest name match to the "
            f"question but was excluded from CONTEXT because it {excluded_top_match['reason']}"
            f" -- it is NOT one of the CONTEXT rows below. This NOTE is itself the answer if "
            f"the guest was asking about this specific dish -- do NOT say you don't have "
            f"information or tell them to ask staff instead; state plainly, using this NOTE, "
            f"why the dish doesn't meet their requirement, rather than declining or answering "
            f"as if a different CONTEXT dish is the one they asked about."
        )
    return text

## 8. End-to-end answer function

`answer()` is the full path: `search()` (LLM understanding + retrieval) -> `tone_for()` from
the understanding step's intent, appended to `GENERATION_SYSTEM_PROMPT` -> prompt assembly ->
`call_llm()` for generation -- and
returns every intermediate artifact, not just the final text. `show_answer()` prints all of
it: what the understanding call extracted, the retrieval verdict, the exact system and user
prompts sent to the generation call, the answer, and what this question actually cost on
**both** APIs the pipeline touches:

- **LLM (Groq)** -- understanding and generation token counts, separately, from each call's
  own `usage` block (not estimated). Both calls use `openai/gpt-oss-120b` today
  (`UNDERSTAND_MODEL`/`GENERATION_MODEL`), but are tracked and configured separately since
  they're free to diverge later.
- **Embedding (Cohere)** -- the rerank call's `search_units`, real billing data from section
  4's `rerank()`. The hybrid query's own vectorization of your question text is *not*
  included -- it happens inside Weaviate's managed integration and Weaviate never reports
  what that internal Cohere call cost, so there is nothing to read for it from this notebook.

Both roll up into `SESSION_USAGE`, so a whole "Run All" shows its cumulative consumption
across every test cell, not just the last question's.

In [ ]:
SESSION_USAGE = {"questions": 0, "total_tokens": 0, "rerank_search_units": 0}


def answer(question: str) -> dict:
    """Full pipeline: search (understand + retrieve) -> build prompt -> generate."""
    result = search(question)
    intent = result["understanding"]["intent"]
    tone = tone_for(intent)
    temperature = temperature_for(intent)
    system_prompt = f"{GENERATION_SYSTEM_PROMPT}\n\n{tone}"
    context = build_context(result["ranked"]) if result["answerable"] else "(no confident match)"
    user_prompt = build_user_prompt(
        question, context, result["relaxed_fields"], result["excluded_top_match"]
    )
    if not LLM_API_KEY:
        reply = "[LLM_API_KEY not set -- skipping live call]"
        gen_usage = _zero_usage()
    else:
        gen = call_llm(
            system_prompt,
            user_prompt,
            model=GENERATION_MODEL,
            temperature=temperature,
            reasoning_effort=GENERATION_REASONING_EFFORT,
        )
        reply = gen["text"]
        gen_usage = gen["usage"]
    understand_usage = result["understanding"]["usage"]
    usage = {
        "understand": understand_usage,
        "generate": gen_usage,
        "total_tokens": understand_usage["total_tokens"] + gen_usage["total_tokens"],
    }
    return {
        "question": question,
        "search": result,
        "intent": intent,
        "tone": tone,
        "temperature": temperature,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "answer": reply,
        "usage": usage,
    }


def show_answer(question: str) -> dict:
    """Run answer() and print the full trace: understanding, retrieval, prompts, and reply."""
    r = answer(question)
    s = r["search"]
    u = s["understanding"]
    usage = r["usage"]
    rerank_units = s["rerank_search_units"]
    SESSION_USAGE["questions"] += 1
    SESSION_USAGE["total_tokens"] += usage["total_tokens"]
    SESSION_USAGE["rerank_search_units"] += rerank_units
    verdict = "ANSWERABLE" if s["answerable"] else "NO CONFIDENT MATCH"
    print(f"q: {question!r}")
    print(
        f"   understanding: intent={u['intent']}  dietary={u['dietary']}  "
        f"price_max_gbp={u['price_max_gbp']}  allergens_exclude={u['allergens_exclude'] or '-'}  "
        f"category_hint={u['category_hint'] or '-'}"
    )
    print(
        f"   gluten_free_only={u['gluten_free_only']}  kcal_max={u['kcal_max']}  "
        f"protein_min_g={u['protein_min_g']}  alcohol_free={u['alcohol_free']}"
    )
    print(f"   search_text: {s['search_text']!r}")
    print(
        f"   retrieval={verdict} (top rr {s['top']:.3f})  "
        f"retrieved {s['retrieved']} -> kept {s['kept']}"
        + (f"  relaxed={s['relaxed_fields']}" if s["relaxed_fields"] else "")
    )
    tone_label = "faq" if r["intent"] == "faq" else "menu"
    print(
        f"   generation tone={tone_label}  temperature={r['temperature']}  "
        f"reasoning_effort={GENERATION_REASONING_EFFORT!r}"
    )
    print(
        f"   LLM usage: understand {usage['understand']['total_tokens']} tok + "
        f"generate {usage['generate']['total_tokens']} tok = {usage['total_tokens']} tok "
        f"for this question  (session so far: {SESSION_USAGE['total_tokens']} tok over "
        f"{SESSION_USAGE['questions']} questions)"
    )
    print(
        f"   embedding usage: rerank {rerank_units} search unit(s) this question  "
        f"(session so far: {SESSION_USAGE['rerank_search_units']}); query vectorization runs "
        f"inside Weaviate and isn't reported back to the client -- not counted here"
    )
    print("\n--- SYSTEM PROMPT (generation) ---")
    print(r["system_prompt"])
    print("\n--- USER PROMPT ---")
    print(r["user_prompt"])
    print("\n--- ANSWER ---")
    print(r["answer"])
    print()
    return r

## 9. Step-by-step pipeline walkthrough

`explain_pipeline()` runs the exact same `answer()` used everywhere else in this notebook --
no new logic, only a different presentation -- and renders every stage of the pipeline as a
numbered, human-readable walkthrough: what each stage was given, what it did, and what it
produced. `show_answer()` (section 8) stays as the compact trace for quick eyeballing; this is
the full narrative for understanding *why* a given answer came out the way it did.

In [ ]:
from IPython.display import Markdown, display


def _fmt_json(d: dict) -> str:
    return "```json\n" + json.dumps(d, indent=2, ensure_ascii=False) + "\n```"


def _describe_filter(u: dict) -> str:
    """Human-readable form of what build_filter(u) turns into a Weaviate filter."""
    clauses = []
    if u["dietary"]:
        clauses.append(f"`dietary_tags` contains `{u['dietary']}`")
    if u["price_max_gbp"] is not None:
        clauses.append(f"`price_gbp` <= {u['price_max_gbp']}")
    if u["gluten_free_only"]:
        clauses.append("`is_gluten_free_listed` is `true`")
    if u["kcal_max"] is not None:
        clauses.append(f"`kcal` <= {u['kcal_max']}")
    if u["protein_min_g"] is not None:
        clauses.append(f"`protein_g` >= {u['protein_min_g']}")
    if u["alcohol_free"]:
        clauses.append("`abv_percent` <= 0.5")
    return "; ".join(clauses) if clauses else "*none -- no hard filter applied, pure hybrid search*"


def _hybrid_score(o) -> float:
    return o.metadata.score or 0.0


def _numbered(items: list[str]) -> str:
    return "\n".join(f"{i}. {text}" for i, text in enumerate(items, start=1))


def explain_pipeline(question: str) -> dict:
    """Run the full pipeline via answer() and render it as a numbered, detailed walkthrough:
    every stage's exact input (including full prompts sent to the LLM), what it did, and its
    output. Purely a presentation layer over the same verified answer()/search() used by
    show_answer() -- no pipeline behavior changes here.
    """
    r = answer(question)
    s = r["search"]
    u = s["understanding"]

    steps: list[str] = []

    def step(title: str, body: str) -> None:
        steps.append(f"### Step {len(steps) + 1} -- {title}\n\n{body}")

    step("Guest asks a question", f"> {question}")

    understanding_view = {k: v for k, v in u.items() if k != "usage"}
    step(
        "Query understanding (LLM call)",
        f"**System prompt sent to `{UNDERSTAND_MODEL}`** "
        f"(`temperature={UNDERSTAND_TEMPERATURE}`, "
        f"`reasoning_effort={UNDERSTAND_REASONING_EFFORT!r}`):\n\n"
        f"```\n{UNDERSTAND_SYSTEM_PROMPT}\n```\n\n"
        f"**User message sent:**\n\n> {question}\n\n"
        f"**Output (parsed JSON):**\n\n{_fmt_json(understanding_view)}\n\n"
        f"*Cost: {u['usage']['total_tokens']} tokens "
        f"({u['usage']['prompt_tokens']} prompt + {u['usage']['completion_tokens']} completion).*",
    )

    if u["intent"] == "faq":
        intent_note = (
            "Classified as an **FAQ** question (house policy: hours, bookings, delivery, "
            "payments, gift cards, allergen-info pointers -- not menu content). "
            "`category_hint` stays empty by design for this intent, and generation will use "
            "**FAQ tone**."
        )
    else:
        intent_note = (
            "Classified as a **menu** question (a dish, ingredient, price, nutrition value, "
            "dish comparison, or a general browse/availability question about the menu). "
            "Generation will use **menu tone**."
        )
    step("Identify question type -- menu item vs. FAQ", intent_note)

    step(
        "Build the retrieval query",
        f"**Search text** (dietary/price/allergy wording already stripped, `category_hint` "
        f'folded in as a soft signal): `"{s["search_text"]}"`\n\n'
        f"**Server-side filter:** {_describe_filter(u)}",
    )

    retrieved_lines = _numbered(
        [f"{line(o)}  (hybrid score: {_hybrid_score(o):.3f})" for o in s["retrieved_objects"]]
    )
    step(
        "Hybrid retrieval against `KnowledgeBase`",
        f"Weaviate hybrid search (`alpha={ALPHA}`, keyword + vector, top `{K}`) with the query "
        f"and filter above returned **{s['retrieved']} candidate row(s)**, in hybrid-score "
        f"order:\n\n{retrieved_lines}",
    )

    if s["excluded"]:
        dropped = [o for o in s["retrieved_objects"] if allergen_set(o) & set(s["excluded"])]
        dropped_lines = (
            _numbered([line(o) for o in dropped])
            if dropped
            else "*(none of the retrieved rows matched)*"
        )
        step(
            "Allergen exclusion",
            f"Rows whose `allergens_contains`/`allergens_may_contain` include any of "
            f"**{', '.join(s['excluded'])}** are dropped before ranking -- "
            f"kept **{s['kept']} of {s['retrieved']}**.\n\n**Dropped:**\n\n{dropped_lines}",
        )

    before_lines = _numbered(
        [f"{line(o)}  (hybrid score: {_hybrid_score(o):.3f})" for o in s["kept_objects"]]
    )
    after_lines = _numbered(
        [
            f"{line(h['obj'])}  (rerank score: {h['rerank']:.3f}, hybrid score: {h['hybrid']:.3f})"
            for h in s["ranked"]
        ]
    )
    step(
        "Rerank (Cohere `rerank-v3.5`) -- before and after",
        f"**Before** -- the {len(s['kept_objects'])} kept candidate(s), still in hybrid-score "
        f"order:\n\n{before_lines}\n\n"
        f"**After** -- Cohere reranks all of them against the search text and keeps the top "
        f"`{TOP_N}`, reordered by actual relevance to the question rather than hybrid score "
        f"alone:\n\n{after_lines}",
    )

    relax_note = ""
    if s["relaxed_fields"]:
        relax_note = (
            "\n\nNo candidate cleared the gate on the first pass, so these constraints were "
            f"progressively relaxed, least essential first, and retrieval re-run: "
            f"**{', '.join(s['relaxed_fields'])}**."
        )
    if s["ranked"]:
        gate_input = _numbered(
            [f"{pstr(h['obj'], 'name')}  (rerank score: {h['rerank']:.3f})" for h in s["ranked"]]
        )
    else:
        gate_input = "*(no candidates survived to be reranked)*"
    step(
        "Answerability gate",
        f"**Input** -- the {len(s['ranked'])} reranked item(s) from the previous step, with "
        f"their rerank scores:\n\n{gate_input}\n\n"
        f"**Score check:** top rerank score **{s['top']:.3f}** vs. required gate `{GATE}`."
        f"{relax_note}\n\n"
        f"**Output:** the question **{'cleared' if s['answerable'] else 'did NOT clear'}** the "
        f"gate -- retrieval is **{'ANSWERABLE' if s['answerable'] else 'NO CONFIDENT MATCH'}**.",
    )

    if s["answerable"]:
        item_names = [pstr(h["obj"], "name") for h in s["ranked"]]
        context_body = (
            f"**{len(item_names)} row(s)** go into CONTEXT for generation:\n\n"
            + "\n".join(f"- {n}" for n in item_names)
        )
    else:
        context_body = (
            'No row cleared the gate, so CONTEXT is the literal string `"(no confident match)"` '
            "-- the system prompt's decline-don't-guess rule fires on exactly this."
        )
    if s["excluded_top_match"]:
        context_body += (
            f"\n\n**NOTE attached:** `{s['excluded_top_match']['name']}` was the closest "
            f"name match in the whole corpus but was excluded because it "
            f"{s['excluded_top_match']['reason']} -- generation is told this explicitly rather "
            "than silently substituting a different dish."
        )
    step("Assemble CONTEXT for generation", context_body)

    tone_label = (
        "FAQ -- warm, conversational" if r["intent"] == "faq" else "menu -- precise, literal"
    )
    step(
        "Determine tone and temperature",
        f"Using the intent classified in step 2 (**`{r['intent']}`**), generation is configured "
        f"with the **{tone_label}** tone at **`temperature={r['temperature']}`** and "
        f"**`reasoning_effort={GENERATION_REASONING_EFFORT!r}`**.\n\n"
        f"**Tone instruction applied** (appended to the fixed system prompt "
        f"below):\n\n> {r['tone']}",
    )

    gen_usage = r["usage"]["generate"]
    step(
        "Generation (LLM call)",
        f"**System prompt sent to `{GENERATION_MODEL}`** (fixed grounding rules + the tone "
        f"instruction from the previous step):\n\n```\n{r['system_prompt']}\n```\n\n"
        f"**User prompt sent** (question + CONTEXT + any NOTEs):\n\n"
        f"```\n{r['user_prompt']}\n```\n\n"
        f"**Output (generated answer):**\n\n{r['answer']}\n\n"
        f"*Cost: {gen_usage['total_tokens']} tokens "
        f"({gen_usage['prompt_tokens']} prompt + {gen_usage['completion_tokens']} completion).*",
    )

    display(Markdown("\n\n---\n\n".join(steps)))
    print(
        f"\nTotal this question: {r['usage']['total_tokens']} tokens "
        f"(understand {r['usage']['understand']['total_tokens']} + "
        f"generate {gen_usage['total_tokens']}), "
        f"{s['rerank_search_units']} Cohere rerank search unit(s)."
    )
    return r

## 10. Demo -- ask your own question

Edit `QUESTION` and re-run this cell as many times as you like -- it reuses the connection
opened in cell 1, so there's no need to re-run the notebook from the top.

In [ ]:
# QUESTION = "a vegan starter under £6"  # <- edit this, then re-run the cell
QUESTION = "is there coffee?"

_ = explain_pipeline(QUESTION)